# Create SQLite Database from CSV Files

This notebook creates a SQLite database named `baseball.db` with three tables based on the provided CSV files: `people.csv`, `teams.csv`, and `Batting(1).csv`. The tables will have correct data types and relational constraints.

In [1]:
import pandas as pd
import sqlite3
from sqlalchemy import create_engine, text

## Load CSV Files into Pandas DataFrames

Read the three CSV files into separate pandas DataFrames with appropriate data types.

In [3]:
# Load the DataFrames
people_df = pd.read_csv('people.csv')
teams_df = pd.read_csv('teams.csv')
batting_df = pd.read_csv('Batting(1).csv')

print("DataFrames loaded successfully!")
print(f"People: {people_df.shape}")
print(f"Teams: {teams_df.shape}")
print(f"Batting: {batting_df.shape}")

DataFrames loaded successfully!
People: (24270, 25)
Teams: (3614, 48)
Batting: (128598, 22)


## Inspect and Adjust Data Types

Check the data types of the loaded DataFrames.

In [7]:
print("People columns:", list(people_df.columns))
print("Teams columns:", list(teams_df.columns))
print("Batting columns:", list(batting_df.columns))

People columns: ['ID', 'playerID', 'birthYear', 'birthMonth', 'birthDay', 'birthCity', 'birthCountry', 'birthState', 'deathYear', 'deathMonth', 'deathDay', 'deathCountry', 'deathState', 'deathCity', 'nameFirst', 'nameLast', 'nameGiven', 'weight', 'height', 'bats', 'throws', 'debut', 'bbrefID', 'finalGame', 'retroID']
Teams columns: ['yearID', 'lgID', 'teamID', 'franchID', 'divID', 'Rank', 'G', 'Ghome', 'W', 'L', 'DivWin', 'WCWin', 'LgWin', 'WSWin', 'R', 'AB', 'H', '2B', '3B', 'HR', 'BB', 'SO', 'SB', 'CS', 'HBP', 'SF', 'RA', 'ER', 'ERA', 'CG', 'SHO', 'SV', 'IPouts', 'HA', 'HRA', 'BBA', 'SOA', 'E', 'DP', 'FP', 'name', 'park', 'attendance', 'BPF', 'PPF', 'teamIDBR', 'teamIDlahman45', 'teamIDretro']
Batting columns: ['playerID', 'yearID', 'stint', 'teamID', 'lgID', 'G', 'AB', 'R', 'H', '2B', '3B', 'HR', 'RBI', 'SB', 'CS', 'BB', 'SO', 'IBB', 'HBP', 'SH', 'SF', 'GIDP']


## Create SQLite Database and Define Table Schemas

Create the SQLite database and define the table schemas with primary and foreign key constraints.

In [3]:
teams_sql_dtypes = {
    'yearID': 'INTEGER',
    'lgID': 'TEXT',
    'teamID': 'TEXT',
    'franchID': 'TEXT',
    'divID': 'TEXT',
    'Rank': 'INTEGER',
    'G': 'INTEGER',
    'Ghome': 'INTEGER',
    'W': 'INTEGER',
    'L': 'INTEGER',
    'DivWin': 'TEXT',
    'WCWin': 'TEXT',
    'LgWin': 'TEXT',
    'WSWin': 'TEXT',
    'R': 'INTEGER',
    'AB': 'INTEGER',
    'H': 'INTEGER',
    '2B': 'INTEGER',
    '3B': 'INTEGER',
    'HR': 'INTEGER',
    'BB': 'INTEGER',
    'SO': 'INTEGER',
    'SB': 'INTEGER',
    'CS': 'INTEGER',
    'HBP': 'INTEGER',
    'SF': 'INTEGER',
    'RA': 'INTEGER',
    'ER': 'INTEGER',
    'ERA': 'REAL',
    'CG': 'INTEGER',
    'SHO': 'INTEGER',
    'SV': 'INTEGER',
    'IPouts': 'INTEGER',
    'HA': 'INTEGER',
    'HRA': 'INTEGER',
    'BBA': 'INTEGER',
    'SOA': 'INTEGER',
    'E': 'INTEGER',
    'DP': 'INTEGER',
    'FP': 'REAL',
    'name': 'TEXT',
    'park': 'TEXT',
    'attendance': 'INTEGER',
    'BPF': 'INTEGER',
    'PPF': 'INTEGER',
    'teamIDBR': 'TEXT',
    'teamIDlahman45': 'TEXT',
    'teamIDretro': 'TEXT'
}

batting_sql_dtypes = {
    'playerID': 'TEXT',
    'yearID': 'INTEGER',
    'stint': 'INTEGER',
    'teamID': 'TEXT',
    'lgID': 'TEXT',
    'G': 'INTEGER',
    'AB': 'INTEGER',
    'R': 'INTEGER',
    'H': 'INTEGER',
    '2B': 'INTEGER',
    '3B': 'INTEGER',
    'HR': 'INTEGER',
    'RBI': 'INTEGER',
    'SB': 'INTEGER',
    'CS': 'INTEGER',
    'BB': 'INTEGER',
    'SO': 'INTEGER',
    'IBB': 'INTEGER',
    'HBP': 'INTEGER',
    'SH': 'INTEGER',
    'SF': 'INTEGER',
    'GIDP': 'INTEGER'
}

## Insert DataFrames into SQLite Tables

Insert the data from each DataFrame into its corresponding SQLite table.

In [9]:
# Insert data into tables
people_df.to_sql('people', engine, if_exists='append', index=False)
teams_df.to_sql('teams', engine, if_exists='append', index=False)
batting_df.to_sql('batting', engine, if_exists='append', index=False)

print("Data inserted successfully!")

Data inserted successfully!


## Verify Data and Constraints in Database

Query the database to confirm that the data has been loaded correctly and that the constraints are in place.

In [10]:
# Verify data insertion
with engine.connect() as conn:
    # Check row counts
    people_count = conn.execute(text('SELECT COUNT(*) FROM people')).fetchone()[0]
    teams_count = conn.execute(text('SELECT COUNT(*) FROM teams')).fetchone()[0]
    batting_count = conn.execute(text('SELECT COUNT(*) FROM batting')).fetchone()[0]
    
    print(f"People table: {people_count} rows")
    print(f"Teams table: {teams_count} rows")
    print(f"Batting table: {batting_count} rows")
    
    # Check a sample query with joins
    sample_query = conn.execute(text('''
        SELECT p.nameFirst, p.nameLast, b.yearID, b.HR, t.name
        FROM batting b
        JOIN people p ON b.playerID = p.playerID
        JOIN teams t ON b.yearID = t.yearID AND b.teamID = t.teamID
        LIMIT 5
    ''')).fetchall()
    
    print("\nSample joined data:")
    for row in sample_query:
        print(row)

print("Database verification complete!")

People table: 24270 rows
Teams table: 3614 rows
Batting table: 128598 rows

Sample joined data:
('David', 'Aardsma', 2004, 0, 'San Francisco Giants')
('David', 'Aardsma', 2006, 0, 'Chicago Cubs')
('David', 'Aardsma', 2007, 0, 'Chicago White Sox')
('David', 'Aardsma', 2008, 0, 'Boston Red Sox')
('David', 'Aardsma', 2009, 0, 'Seattle Mariners')
Database verification complete!


In [2]:
# Test the FastAPI /years endpoint using TestClient
from fastapi.testclient import TestClient
import main

client = TestClient(main.app)
response = client.get("/years")
print("/years status:", response.status_code)
print("/years json sample:", response.json() if response.status_code == 200 else response.text)

/years status: 200
/years json sample: {'years': [1871, 1872, 1873, 1874, 1875, 1876, 1877, 1878, 1879, 1880, 1881, 1882, 1883, 1884, 1885, 1886, 1887, 1888, 1889, 1890, 1891, 1892, 1893, 1894, 1895, 1896, 1897, 1898, 1899, 1900, 1901, 1902, 1903, 1904, 1905, 1906, 1907, 1908, 1909, 1910, 1911, 1912, 1913, 1914, 1915, 1916, 1917, 1918, 1919, 1920, 1921, 1922, 1923, 1924, 1925, 1926, 1927, 1928, 1929, 1930, 1931, 1932, 1933, 1934, 1935, 1936, 1937, 1938, 1939, 1940, 1941, 1942, 1943, 1944, 1945, 1946, 1947, 1948, 1949, 1950, 1951, 1952, 1953, 1954, 1955, 1956, 1957, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]}


Bad pipe message: %s [b'0.9,*/*;q=0.8\r\nHost: localhost:35133\r\nUser-Agent: Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/60', b'1.15 (KHTML, like Gecko) Version/17.5 Safari/605.1.1']
Bad pipe message: %s [b'\nAccept-Encoding: gzip, deflate, br\r\nAccept-Language', b'en-US,en;q=0.9\r\nReferer: https://github.com/\r\nX-Request-I']
Bad pipe message: %s [b' 431bec217a1506d204178089ca1afff7\r\nX-Real-IP: 66.211.206.59\r\nX-Forw']
Bad pipe message: %s [b'ded-Port: 443\r\nX-Forwarded-Scheme: https\r\nX-Original-URI: /\r\nX-Scheme: https\r\nsec-fetch-site: cr']
Bad pipe message: %s [b's-site\r\nsec-fetch-dest: document\r\nsec-fetch-mode: navigate\r\nX-Forwarded-Proto: https\r\nX-Forwarded-Host: sturdy', b'ylophone-5gp745qww74p2pr7-35133.app.github.d']
Bad pipe message: %s [b' Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.5 Safari/605.1']
Bad pipe message: %s [b'5\r\nAccept-Encoding: gzip, deflate, br\r\nAccept', b'anguage: 

In [3]:
# Test that /teams returns teams for a given year
response = client.get("/teams?year=2000")
print("/teams status:", response.status_code)
print("/teams json sample:", response.json() if response.status_code == 200 else response.text)


/teams status: 404
/teams json sample: {"detail":"Not Found"}


In [5]:
# Reload the FastAPI app after code changes and re-test endpoints
import importlib
import main

importlib.reload(main)

client = TestClient(main.app)

response = client.get("/years")
print("/years status:", response.status_code)
print("/years json sample:", response.json() if response.status_code == 200 else response.text)

response = client.get("/teams?year=2000")
print("/teams status:", response.status_code)
print("/teams json sample:", response.json() if response.status_code == 200 else response.text)


/years status: 200
/years json sample: {'years': [1871, 1872, 1873, 1874, 1875, 1876, 1877, 1878, 1879, 1880, 1881, 1882, 1883, 1884, 1885, 1886, 1887, 1888, 1889, 1890, 1891, 1892, 1893, 1894, 1895, 1896, 1897, 1898, 1899, 1900, 1901, 1902, 1903, 1904, 1905, 1906, 1907, 1908, 1909, 1910, 1911, 1912, 1913, 1914, 1915, 1916, 1917, 1918, 1919, 1920, 1921, 1922, 1923, 1924, 1925, 1926, 1927, 1928, 1929, 1930, 1931, 1932, 1933, 1934, 1935, 1936, 1937, 1938, 1939, 1940, 1941, 1942, 1943, 1944, 1945, 1946, 1947, 1948, 1949, 1950, 1951, 1952, 1953, 1954, 1955, 1956, 1957, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]}
/teams status: 200


In [1]:
import inspect

import main

print("main module file:", main.__file__)
print("get_years source:\n", inspect.getsource(main.get_years))


main module file: /workspaces/baseball_app/main.py
get_years source:
 @app.get("/years")
async def get_years():
    """Return a sorted list of all years available in the teams table."""
    with get_session() as session:
        statement = select(Teams.yearID).distinct().order_by(Teams.yearID)
        years = session.exec(statement).all()

    return {"years": years}



In [6]:
# Verify the endpoints return expected shapes without printing everything
import importlib
import main

importlib.reload(main)
client = TestClient(main.app)

for path in ["/years", "/teams?year=2000"]:
    r = client.get(path)
    print(path, "status", r.status_code)
    try:
        data = r.json()
        if isinstance(data, dict) and "years" in data:
            print("  years count", len(data["years"]))
        if isinstance(data, dict) and "teams" in data:
            print("  teams count", len(data["teams"]))
            print("  sample", data["teams"][0:5])
    except Exception as e:
        print("  parse error", e)


/years status 200
  years count 155
/teams?year=2000 status 200
  teams count 30
  sample [{'teamID': 'ANA', 'name': 'Anaheim Angels'}, {'teamID': 'ARI', 'name': 'Arizona Diamondbacks'}, {'teamID': 'ATL', 'name': 'Atlanta Braves'}, {'teamID': 'BAL', 'name': 'Baltimore Orioles'}, {'teamID': 'BOS', 'name': 'Boston Red Sox'}]
